# 03 · Data and scoring (stage S1, GQA + TextVQA) + merged evaluation runner

Checks:
1. **Splits:** GQA test-dev split into test / ctrl_train / dev / lora_train with the sizes in `configs/mvp.yaml`; no image in two splits; manifest hash stable.
2. **Items:** dev items load as `Item` with image, question, answer, question type.
3. **Extraction + scoring:** hand-written outputs give the expected answer and score.
4. **TextVQA:** image-disjoint splits, items with 10 answers, VQA accuracy (lmms-eval answer processing). No model needed.
5. **Evaluation runner** (merged from Jiaqi's repo): `N_ITEMS` GQA dev items at R=3, full tokens; rows carry provenance; re-running resumes without new work.

Thinking is still on here (stage S2 turns it off), so the parse rate is shown, not asserted.

In [ ]:
N_ITEMS = 20
R = 3
CONFIG = "configs/mvp.yaml"

In [ ]:
from adaloop import api
ROOT, cfg = api.setup_notebook(CONFIG)

CHECKS = []


def check(name, ok, detail=""):
    CHECKS.append((name, bool(ok)))
    print(("PASS " if ok else "FAIL ") + f"{name}" + (f": {detail}" if detail else ""))
    assert ok, f"{name} failed: {detail}"


print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

## 1 · Splits

In [ ]:
from collections import Counter

print("[STAGE] splits")
manifest = api.build_gqa_splits(cfg, ROOT)
sizes = {name: len(ids) for name, ids in manifest["splits"].items()}
n_images = {name: len(ids) for name, ids in manifest["images"].items()}
print("questions per split", sizes)
print("images per split   ", n_images)

for name, want in cfg["splits"]["sizes"].items():
    if want is not None:
        check(f"S1 split size {name}", sizes[name] == want, f"{sizes[name]} vs {want}")
names = list(manifest["splits"])
pairs = [(a, b) for i, a in enumerate(names) for b in names[i + 1:]]
check("S1 question-disjoint", all(not set(manifest["splits"][a]) & set(manifest["splits"][b]) for a, b in pairs))
check("S1 image-disjoint", all(not set(manifest["images"][a]) & set(manifest["images"][b]) for a, b in pairs))
check("S1 manifest hash stable", api.load_manifest(ROOT / cfg["benchmarks"]["gqa"]["manifest"])["hash"] == manifest["hash"],
      manifest["hash"])
print("✓ splits")

## 2 · Items

In [ ]:
print("[STAGE] items")
items = api.load_gqa(cfg, "dev", limit=N_ITEMS, root=ROOT)
for it in items[:5]:
    print(f"{it.item_id:>10} {it.question_type:8} {it.image.size} {it.question!r} -> {it.answers}")
print("question types", Counter(it.question_type for it in items))

check("S1 item count", len(items) == N_ITEMS, len(items))
check("S1 items have answers", all(it.answers and it.answers[0] for it in items))
check("S1 items are dev and in manifest", all(it.split == "dev" and it.item_id in manifest["splits"]["dev"] for it in items))
n_item_images = len({it.image_id for it in items})
print("distinct image_id", n_item_images)
check("S1 items span several images", n_item_images > 1, n_item_images)
print("✓ items")

In [ ]:
items[0].image

## 3 · Answer extraction and scoring

In [ ]:
print("[STAGE] extraction + scoring")
cases = [  # (raw output, hit_limit, expected answer)
    ("<|content_text|>Yes<|end_message|>", False, "Yes"),
    ("<|content_thinking|>the sky is blue<|content_text|>Answer: blue.<|end_message|>", False, "blue."),
    ("<|content_thinking|>still thinking about it", True, None),
    ("<|content_text|>a long answer that got cut", True, None),
    ("left", False, "left"),
]
for raw, hit, want in cases:
    got = api.extract_answer(raw, hit)
    print(f"{raw[:60]!r:64} -> {got!r}")
    check(f"S1 extract {want!r}", got == want, f"got {got!r}")

check("S1 gqa exact match", api.score_gqa("Blue.", ["blue"]) == 1.0)
check("S1 gqa mismatch", api.score_gqa("dark blue", ["blue"]) == 0.0)
check("S1 gqa unparsed", api.score_gqa(None, ["blue"]) == 0.0)
print("✓ extraction + scoring")

## 4 · TextVQA: splits, items, VQA accuracy

In [ ]:
print("[STAGE] textvqa splits")
tv_manifest = api.build_textvqa_splits(cfg, ROOT)
tv_sizes = {name: len(ids) for name, ids in tv_manifest["splits"].items()}
tv_images = {name: set(ids) for name, ids in tv_manifest["images"].items()}
print("questions per split", tv_sizes)
print("images per split   ", {name: len(ids) for name, ids in tv_images.items()})

for name, want in cfg["splits"]["sizes"].items():
    if want is not None:
        check(f"S1 textvqa split size {name}", tv_sizes[name] == want, f"{tv_sizes[name]} vs {want}")
check("S1 textvqa image-disjoint", sum(map(len, tv_images.values())) == len(set().union(*tv_images.values())))
print("✓ textvqa splits")

In [ ]:
print("[STAGE] textvqa items")
tv_items = api.load_textvqa(cfg, "dev", limit=N_ITEMS, root=ROOT)
for it in tv_items[:3]:
    print(f"{it.item_id:>6} {it.image.size} {it.question!r}\n       {it.answers}")

check("S1 textvqa item count", len(tv_items) == N_ITEMS, len(tv_items))
check("S1 textvqa 10 answers per item", all(len(it.answers) == 10 for it in tv_items))
print("✓ textvqa items")

In [ ]:
tv_items[0].image

In [ ]:
print("[STAGE] textvqa scoring")
cases = [  # (name, prediction, 10 reference answers, expected score)
    ("all match", "stop", ["stop"] * 10, 1.0),
    ("1 of 10 match", "stop", ["stop"] + ["go"] * 9, 0.3),
    ("no match", "left", ["stop"] * 10, 0.0),
    ("article + punctuation", "The Coca-Cola.", ["coca cola"] * 10, 1.0),
    ("unparsed", None, ["stop"] * 10, 0.0),
]
for name, pred, answers, want in cases:
    got = api.score_textvqa(pred, answers)
    check(f"S1 textvqa {name}", abs(got - want) < 1e-9, f"{pred!r} -> {got}")
print("✓ textvqa scoring")

## 5 · Evaluation runner (R=3, full tokens)

In [ ]:
print("[STAGE] evaluation")
vlm = api.load_model(cfg)
out = ROOT / cfg["out_dir"] / "nb03" / f"gqa_dev_n{N_ITEMS}_res{cfg['res_px']}_R{R}.jsonl"
summary = api.run_evaluation(vlm, items, cfg, R, out, resume=True)
summary

In [ ]:
import pandas as pd

rows = api.read_results(out)
df = pd.DataFrame(rows)
display(df[["item_id", "question_type", "pred", "answers", "correct", "parse_ok", "n_vis", "n_out_tokens", "generate_s"]].head(10))
print("raw output example:", repr(rows[0]["raw_output"][:300]))

check("S1 one row per item", len(rows) == N_ITEMS and df.item_id.nunique() == N_ITEMS, len(rows))
check("S1 rows carry provenance", all(r.get(k) for r in rows for k in ("config_hash", "git_hash", "gpu_type")))
check("S1 all items full tokens at R", (df.k == 1.0).all() and (df.R == R).all())
print(f"accuracy {summary['accuracy']:.1f}  parse rate {summary['parse_rate']:.1f}  (thinking still on; S2 fixes parsing)")

In [ ]:
print("[STAGE] resume")
again = api.run_evaluation(vlm, items, cfg, R, out, resume=True)
check("S1 resume adds no rows", len(api.read_results(out)) == N_ITEMS)
check("S1 resume same accuracy", again["accuracy"] == summary["accuracy"])
print("✓ evaluation")

## Summary

In [ ]:
pd.DataFrame(CHECKS, columns=["check", "pass"])